# VOC → YOLO

Converts a Pascal-VOC `VOC_PCB/` tree into `PCB_DATA_YOLO/` (images, labels, `data.yaml`) using the official train / val / test split files. Skip this notebook if you already downloaded the YOLO archive.

In [ ]:
from pathlib import Path
import shutil
import xml.etree.ElementTree as ET

SRC = Path("./VOC_PCB")
DST = Path("./PCB_DATA_YOLO")
CLEAN_DEST_IF_EXISTS = False

CLASSES = [
    "missing_hole",
    "mouse_bite",
    "open_circuit",
    "short",
    "spur",
    "spurious_copper",
]

# Variant spellings found in the source XMLs -> canonical class id.
CLASS_ALIASES = {
    "missing hole": "missing_hole",
    "mouse bite": "mouse_bite",
    "open circuit": "open_circuit",
    "spurious copper": "spurious_copper",
}
# Junk / typo tag seen in a few PKU XMLs; drop those objects.
IGNORE_LABELS = {"idaneel"}
IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG", ".BMP"]

print(f"SRC: {SRC.resolve()}")
print(f"DST: {DST.resolve()}")

if not SRC.exists():
    raise FileNotFoundError(f"Source folder not found: {SRC.resolve()}")


In [ ]:
def normalize_label(name: str) -> str:
    name = (name or "").strip()
    return CLASS_ALIASES.get(name, name)


def normalize_image_id(raw_id: str) -> str:
    # Split file lines may be "sample", "sample.jpg" or "JPEGImages/sample.jpg".
    val = raw_id.strip().replace("\\", "/").split("/")[-1]
    return Path(val).stem


def find_image(images_dir: Path, image_id: str):
    for ext in IMG_EXTS:
        p = images_dir / f"{image_id}{ext}"
        if p.exists():
            return p
    return None


def voc_to_yolo_box(img_w, img_h, xmin, ymin, xmax, ymax):
    xmin = max(0.0, min(xmin, img_w - 1))
    xmax = max(0.0, min(xmax, img_w - 1))
    ymin = max(0.0, min(ymin, img_h - 1))
    ymax = max(0.0, min(ymax, img_h - 1))
    bw = max(0.0, xmax - xmin)
    bh = max(0.0, ymax - ymin)
    xc = xmin + bw / 2.0
    yc = ymin + bh / 2.0
    return xc / img_w, yc / img_h, bw / img_w, bh / img_h


def xml_to_yolo_lines(xml_path: Path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    if size is None:
        return []
    w_tag = size.find("width")
    h_tag = size.find("height")
    if w_tag is None or h_tag is None:
        return []
    img_w = float(w_tag.text)
    img_h = float(h_tag.text)
    if img_w <= 0 or img_h <= 0:
        return []

    lines = []
    for obj in root.findall("object"):
        n = obj.find("name")
        b = obj.find("bndbox")
        if n is None or b is None:
            continue
        label = normalize_label(n.text)
        if label in IGNORE_LABELS or label not in CLASSES:
            continue
        xmin_tag = b.find("xmin"); ymin_tag = b.find("ymin")
        xmax_tag = b.find("xmax"); ymax_tag = b.find("ymax")
        if None in (xmin_tag, ymin_tag, xmax_tag, ymax_tag):
            continue
        xmin = float(xmin_tag.text); ymin = float(ymin_tag.text)
        xmax = float(xmax_tag.text); ymax = float(ymax_tag.text)
        xc, yc, bw, bh = voc_to_yolo_box(img_w, img_h, xmin, ymin, xmax, ymax)
        if bw <= 0.0 or bh <= 0.0:
            continue
        lines.append(f"{CLASSES.index(label)} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    return lines


def read_split_ids(split_file: Path):
    if not split_file.exists():
        return []
    return [normalize_image_id(ln) for ln in split_file.read_text(encoding="utf-8").splitlines() if ln.strip()]


images_src = SRC / "JPEGImages"
ann_src    = SRC / "Annotations"
split_src  = SRC / "ImageSets" / "Main"
for d in (images_src, ann_src, split_src):
    if not d.exists():
        raise FileNotFoundError(f"Missing folder: {d.resolve()}")

if DST.exists() and CLEAN_DEST_IF_EXISTS:
    shutil.rmtree(DST)
for sp in ["train", "val", "test"]:
    (DST / "images" / sp).mkdir(parents=True, exist_ok=True)
    (DST / "labels" / sp).mkdir(parents=True, exist_ok=True)

splits = {
    "train": read_split_ids(split_src / "train.txt"),
    "val":   read_split_ids(split_src / "val.txt"),
    "test":  read_split_ids(split_src / "test.txt"),
}
print("Split sizes:", {k: len(v) for k, v in splits.items()})
if sum(len(v) for v in splits.values()) == 0:
    raise RuntimeError("Empty split lists; check VOC paths and the train/val/test files.")

stats = {"converted": 0, "missing_image": 0, "missing_xml": 0, "empty_label": 0}
for split_name, ids in splits.items():
    for image_id in ids:
        img_path = find_image(images_src, image_id)
        xml_path = ann_src / f"{image_id}.xml"
        if img_path is None:
            stats["missing_image"] += 1
            continue
        if not xml_path.exists():
            stats["missing_xml"] += 1
            continue
        yolo_lines = xml_to_yolo_lines(xml_path)
        if not yolo_lines:
            stats["empty_label"] += 1
        shutil.copy2(img_path, DST / "images" / split_name / img_path.name)
        (DST / "labels" / split_name / f"{image_id}.txt").write_text(
            "\n".join(yolo_lines), encoding="utf-8"
        )
        stats["converted"] += 1

if stats["converted"] == 0:
    raise RuntimeError("0 files converted; check that split entries match image filenames.")

names_str = ", ".join([f"'{c}'" for c in CLASSES])
(DST / "data.yaml").write_text(
    f"path: {DST.as_posix()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n\n"
    f"nc: {len(CLASSES)}\n"
    f"names: [{names_str}]\n",
    encoding="utf-8",
)
print("Conversion done.")


In [ ]:
for sp in ["train", "val", "test"]:
    n_img = len(list((DST / "images" / sp).glob("*")))
    n_lbl = len(list((DST / "labels" / sp).glob("*.txt")))
    print(f"{sp:5s} | images: {n_img:5d} | labels: {n_lbl:5d}")
print(f"converted={stats['converted']}  missing_image={stats['missing_image']}  "
      f"missing_xml={stats['missing_xml']}  empty_label={stats['empty_label']}")
print("data.yaml:", (DST / "data.yaml").as_posix())